# EDA and Preprocessing for Gemstone Dataset
This notebook loads the `cubic_zirconia.csv` dataset, performs EDA (saving plots to `eda_plots/`), and calls our preprocessing script.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Ensure src is in path
sys.path.append(os.path.abspath('../../'))
from src.config import CUT_ORDER, COLOR_ORDER, CLARITY_ORDER
from src.preprocess_gemstone import preprocess_cubic_zirconia

# Setup plots directory
PLOT_DIR = './eda_plots/'
os.makedirs(PLOT_DIR, exist_ok=True)

## Part 1: Data Understanding

In [ ]:
df = pd.read_csv('../../data/cubic_zirconia.csv')

# Drop redundant index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nDescribe:\n", df.describe())

In [ ]:
print("Missing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Identify obviously invalid rows (x, y, z, or depth == 0, or price <= 0)
invalid_mask = (df['x'] == 0) | (df['y'] == 0) | (df['z'] == 0) | (df['depth'] == 0) | (df['price'] <= 0)
print(f"Number of invalid rows: {invalid_mask.sum()}")
if invalid_mask.sum() > 0:
    display(df[invalid_mask].head())

## Visualizations

In [ ]:
# Price distribution histogram
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], bins=50, kde=True)
plt.title('Price Distribution')
plt.savefig(os.path.join(PLOT_DIR, 'price_distribution.png'))
plt.show()

In [ ]:
# Carat vs Price scatterplot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='carat', y='price', alpha=0.3)
plt.title('Carat vs Price')
plt.savefig(os.path.join(PLOT_DIR, 'carat_vs_price.png'))
plt.show()

In [ ]:
# Boxplots for carat, depth, and table
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(y=df['carat'], ax=axes[0]).set_title('Carat Boxplot')
sns.boxplot(y=df['depth'], ax=axes[1]).set_title('Depth Boxplot')
sns.boxplot(y=df['table'], ax=axes[2]).set_title('Table Boxplot')
plt.savefig(os.path.join(PLOT_DIR, 'boxplots.png'))
plt.show()

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.savefig(os.path.join(PLOT_DIR, 'correlation_heatmap.png'))
plt.show()

## Part 2: Preprocessing Pipeline
Now we'll call our preprocessing script which implements the cleaning steps (duplicates, invalid rows, depth median imputation, IQR outlier removal for carat/price, and ordinal encoding) and saves to `gemstone_clean.csv`.

In [ ]:
# Run the preprocessing script function
df_clean = preprocess_cubic_zirconia(input_path='../../data/cubic_zirconia.csv', output_path='../../data/gemstone_clean.csv')

print("\nCleaned Data Sample:")
display(df_clean.head())